# Cross-table checks
            Goal: check that the tables match each other. Issues are only found and logged here
            (docs/data_issues.md); they are fixed later in PostgreSQL.

## 0. Setup

In [1]:
import pandas as pd
import numpy as np

customers = pd.read_csv("../data/raw/customers.csv", parse_dates = ['signup_date'], dtype = {'birth_year': 'Int64'})
products = pd.read_csv("../data/raw/products.csv", parse_dates = ['launch_date', 'discontinued_date'])
orders = pd.read_csv("../data/raw/orders.csv", parse_dates = ['order_ts', 'promised_date', 'delivered_date'], dtype = {'session_id' : 'Int64'})
orders.insert(4, "order_date_local", orders["order_ts"].dt.tz_localize("UTC").dt.tz_convert("Europe/Vilnius").dt.tz_localize(None).dt.normalize())
order_items = pd.read_csv("../data/raw/order_items.csv", parse_dates = ['return_date'])
user_events = pd.read_csv("../data/raw/user_events.csv", parse_dates = ['event_ts'], dtype = {'customer_id': 'Int64', 'product_id': 'Int64'})
user_events.insert(6, "event_date_local", user_events["event_ts"].dt.tz_localize("UTC").dt.tz_convert("Europe/Vilnius").dt.tz_localize(None).dt.normalize())
marketing_spend = pd.read_csv("../data/raw/marketing_spend.csv", parse_dates = ['spend_date'], dtype = {'platform_conversions' : 'Int64'})

## Cross-table questions

        ### A. Keys – does every ID have a match in the other table?
        - A1. order_items.order_id → orders.order_id
        - A2. orders.order_id → order_items.order_id
        - A3. order_items.product_id → products.product_id
        - A4. orders.customer_id → customers.customer_id
        - A5. orders.session_id → user_events.session_id
        - A6. user_events.product_id → products.product_id
        
        ### B. Same thing in two tables – do they match?
        - B1. Does order_total = sum of order lines + shipping_fee?
        - B2. Do large quantities (DQ-12) match order_total?
        - B3. Why do purchase events not match orders (DQ-15)?
        
        ### C. Time order – are dates in a logical order?
        - C1. Were products sold before launch_date or after discontinued_date?
        - C2. Is return_date after delivered_date?
        
        ### D. Status
        - D1. Are returned lines only in completed orders?

## A. Keys

### A1. Does every order_id in order_items exist in orders?

In [2]:
order_items_orders = order_items.merge(orders, how = "left_anti", on = "order_id")
print(len(order_items_orders))


0


**Answer:** Yes. 0 order lines without an order.

### A2. Does every order_id in orders exist in order_items?

In [3]:
orders_order_items = orders.merge(order_items, how = "left_anti", on = "order_id")
print(len(orders_order_items))

0


**Answer:** Yes. 0 orders without order lines.

### A3. Does every product_id in order_items exist in products?

In [4]:
order_items_products = order_items.merge(products, how = "left_anti", on = "product_id")
print(len(order_items_products))

0


**Answer:** Yes. 0 order lines with an unknown product.

### A4. Does every customer_id in orders exist in customers?

In [5]:
orders_customer_id = orders.merge(customers, how = "left_anti", on = "customer_id")
print(len(orders_customer_id))

0


**Answer:** Yes. 0 orders with unknown customer.

### A5. Does every session_id in orders exist in user_events?

In [6]:
orders_user_events = orders.dropna(subset=["session_id"]).merge(user_events, how = "left_anti", on = "session_id")
print(len(orders_user_events))

0


**Answer:** Yes. 584 orders have no session_id (expected, see data dictionary), so they were left out. All other orders have a session_id that exists in user_events.

### A6. Does every product_id in user_events exist in products?

In [7]:
user_events_products = user_events.dropna(subset=["product_id"]).merge(products, how = "left_anti", on = "product_id")
print(len(user_events_products))

0


**Answer:** Yes. 34616 user_events have no product_id (expected, see data dictionary), so they were left out. All other user_events have a product_id that exists in products.

## B. Same thing in two tables

### B1. Does order_total = sum of order lines + shipping_fee?

In [8]:
order_items_copy = order_items.copy()
order_items_copy["total_paid"] = round(order_items_copy["quantity"] * order_items_copy["unit_price"] * (1 - (order_items_copy["discount_pct"] / 100)), 2)
order_items_grouped = order_items_copy.groupby(by = "order_id")["total_paid"].sum().reset_index()
print(f"lines after grouping: {len(order_items_grouped)}")

check_order_total = pd.merge(orders, order_items_grouped[["order_id", "total_paid"]], how = "left", on = "order_id")

print(f"order_total lines that don't match sum of order lines + shipping_fee: {len(check_order_total[check_order_total["order_total"] != round((check_order_total["total_paid"] + check_order_total["shipping_fee"]), 2)])}")

not_matching_cases = check_order_total[check_order_total["order_total"] != round((check_order_total["total_paid"] + check_order_total["shipping_fee"]), 2)]
not_matching_cases["difference"] = not_matching_cases["order_total"] - round((not_matching_cases["total_paid"] + not_matching_cases["shipping_fee"]), 2)
display(not_matching_cases.sort_values(by = "difference"))


lines after grouping: 10259
order_total lines that don't match sum of order lines + shipping_fee: 44


,order_id,customer_id,session_id,order_ts,order_date_local,status,coupon_code,payment_method,delivery_method,shipping_fee,order_total,promised_date,delivered_date,total_paid,difference
5515,5516,1271,114965,2025-10-10 20:12:01,2025-10-10,completed,NaN,paypal,courier,0.00,98.97,2025-10-14,2025-10-14,368.88,-269.91
5557,5558,4080,<NA>,2025-10-13 19:21:07,2025-10-13,completed,NaN,card,courier,5.49,24.48,2025-10-15,2025-10-15,189.90,-170.91
5707,5708,2962,118941,2025-10-28 05:21:59,2025-10-28,completed,NaN,bank_link,parcel_locker,0.00,64.97,2025-10-29,2025-10-29,199.88,-134.91
10083,10084,7377,205967,2026-09-17 16:29:35,2026-09-17,completed,WELCOME10,bank_link,courier,0.00,186.29,2026-09-21,2026-09-21,206.99,-20.70
6630,6631,5113,135641,2025-12-20 08:25:32,2025-12-20,completed,NaN,bank_link,parcel_locker,0.00,124.99,2025-12-23,2025-12-23,139.99,-15.00
8200,8201,6151,166442,2026-04-26 17:08:48,2026-04-26,completed,WELCOME10,bank_link,parcel_locker,3.49,18.17,2026-04-28,2026-04-28,29.68,-15.00
4577,4578,3694,94614,2025-07-20 05:41:28,2025-07-20,completed,NaN,card,pickup,0.00,116.99,2025-07-21,2025-07-21,129.99,-13.00
7957,7958,6002,162085,2026-04-09 16:01:47,2026-04-09,completed,NaN,card,courier,0.00,116.06,2026-04-13,2026-04-13,128.96,-12.90
2407,2408,1598,48866,2024-12-21 15:07:54,2024-12-21,completed,NaN,pay_later,courier,0.00,104.98,2024-12-27,2024-12-27,114.98,-10.00
795,796,731,16346,2024-06-30 17:31:53,2024-06-30,completed,NaN,paypal,parcel_locker,0.00,64.98,2024-07-02,2024-07-02,74.98,-10.00


In [9]:
### Test 3 ways to calculate the order sum

def count_not_matching(line_totals, truncate = False):
    df = order_items[["order_id"]].copy()
    df["line_total"] = line_totals
    grouped = df.groupby("order_id")["line_total"].sum().reset_index()
    if truncate:
        grouped["line_total"] = np.floor(round(grouped["line_total"] * 100, 6)) / 100
    check = pd.merge(orders, grouped, how = "left", on = "order_id")
    return len(check[check["order_total"] != round(check["line_total"] + check["shipping_fee"], 2)])

qty = order_items["quantity"]
price = order_items["unit_price"] * (1 - order_items["discount_pct"] / 100)

print("Round each line:", count_not_matching(round(qty * price, 2)))
print("Round unit price:", count_not_matching(qty * round(price, 2)))
print("Truncate order sum:", count_not_matching(qty * price, truncate = True))

Round each line: 44
Round unit price: 91
Truncate order sum: 113


In [10]:
### Look at the orders that are still 0.01 € off

not_matching_001 = not_matching_cases[round(not_matching_cases["difference"], 2) == -0.01]["order_id"]
diff_lines = order_items_copy[order_items_copy["order_id"].isin(not_matching_001)]
diff_lines["line_total_raw"] = diff_lines["quantity"] * diff_lines["unit_price"] * (1 - diff_lines["discount_pct"] / 100)

print(f"Orders still 0.01 € off: {len(not_matching_001)}")
display(diff_lines[["order_id", "quantity", "unit_price", "discount_pct", "line_total_raw", "total_paid"]])

Orders still 0.01 € off: 17


,order_id,quantity,unit_price,discount_pct,line_total_raw,total_paid
1421,826,1,17.99,0,17.9900,17.99
1422,826,1,29.99,50,14.9950,15.00
1423,826,1,22.99,0,22.9900,22.99
1463,848,1,29.99,50,14.9950,15.00
1522,885,1,49.99,0,49.9900,49.99
1523,885,1,14.99,0,14.9900,14.99
1524,885,1,29.99,0,29.9900,29.99
1525,885,1,29.99,50,14.9950,15.00
1526,885,1,29.99,30,20.9930,20.99
1533,890,1,29.99,50,14.9950,15.00


**Rounding check:**
- I tried three ways to get the order sum:
  - round each line to cents, then add them up -> 44 orders don't match
  - round the unit price after discount, then multiply by quantity -> 91 orders don't match
  - add up the lines, then cut the sum down to cents -> 113 orders don't match
- Rounding each line works best. It also makes sense: on a receipt every line is shown in cents, and the total is the sum of those lines.
- 17 orders are still 1 cent off. Their line sums end in exactly half a cent (for example 14.995), so this is a rounding issue, not a data error.
- Together it is only 0.17 €, so from now on I allow a 0.01 € difference when I compare money.
- I don't know the exact rounding rule of the store. In a real job I would ask the IT team.

### Orders that don't match after allowing 0.01 € difference

In [11]:
not_matching_tolerance = not_matching_cases[abs(round(not_matching_cases["difference"], 2)) > 0.01]
print(len(not_matching_tolerance))
display(not_matching_tolerance.sort_values("difference"))

27


,order_id,customer_id,session_id,order_ts,order_date_local,status,coupon_code,payment_method,delivery_method,shipping_fee,order_total,promised_date,delivered_date,total_paid,difference
5515,5516,1271,114965,2025-10-10 20:12:01,2025-10-10,completed,NaN,paypal,courier,0.00,98.97,2025-10-14,2025-10-14,368.88,-269.91
5557,5558,4080,<NA>,2025-10-13 19:21:07,2025-10-13,completed,NaN,card,courier,5.49,24.48,2025-10-15,2025-10-15,189.90,-170.91
5707,5708,2962,118941,2025-10-28 05:21:59,2025-10-28,completed,NaN,bank_link,parcel_locker,0.00,64.97,2025-10-29,2025-10-29,199.88,-134.91
10083,10084,7377,205967,2026-09-17 16:29:35,2026-09-17,completed,WELCOME10,bank_link,courier,0.00,186.29,2026-09-21,2026-09-21,206.99,-20.70
6630,6631,5113,135641,2025-12-20 08:25:32,2025-12-20,completed,NaN,bank_link,parcel_locker,0.00,124.99,2025-12-23,2025-12-23,139.99,-15.00
8200,8201,6151,166442,2026-04-26 17:08:48,2026-04-26,completed,WELCOME10,bank_link,parcel_locker,3.49,18.17,2026-04-28,2026-04-28,29.68,-15.00
4577,4578,3694,94614,2025-07-20 05:41:28,2025-07-20,completed,NaN,card,pickup,0.00,116.99,2025-07-21,2025-07-21,129.99,-13.00
7957,7958,6002,162085,2026-04-09 16:01:47,2026-04-09,completed,NaN,card,courier,0.00,116.06,2026-04-13,2026-04-13,128.96,-12.90
4500,4501,2405,92938,2025-07-12 17:32:56,2025-07-12,completed,NaN,bank_link,parcel_locker,3.49,21.48,2025-07-15,2025-07-15,27.99,-10.00
795,796,731,16346,2024-06-30 17:31:53,2024-06-30,completed,NaN,paypal,parcel_locker,0.00,64.98,2024-07-02,2024-07-02,74.98,-10.00


### Round number differences

In [12]:
round_differences = not_matching_tolerance[round(round(not_matching_tolerance["difference"], 0), 2) == round(not_matching_tolerance["difference"], 2)]
print(f"Orders with a round number difference: {len(round_differences)}")
display(round_differences.sort_values("difference"))

Orders with a round number difference: 16


,order_id,customer_id,session_id,order_ts,order_date_local,status,coupon_code,payment_method,delivery_method,shipping_fee,order_total,promised_date,delivered_date,total_paid,difference
6630,6631,5113,135641,2025-12-20 08:25:32,2025-12-20,completed,NaN,bank_link,parcel_locker,0.00,124.99,2025-12-23,2025-12-23,139.99,-15.0
8200,8201,6151,166442,2026-04-26 17:08:48,2026-04-26,completed,WELCOME10,bank_link,parcel_locker,3.49,18.17,2026-04-28,2026-04-28,29.68,-15.0
4577,4578,3694,94614,2025-07-20 05:41:28,2025-07-20,completed,NaN,card,pickup,0.00,116.99,2025-07-21,2025-07-21,129.99,-13.0
795,796,731,16346,2024-06-30 17:31:53,2024-06-30,completed,NaN,paypal,parcel_locker,0.00,64.98,2024-07-02,2024-07-02,74.98,-10.0
4500,4501,2405,92938,2025-07-12 17:32:56,2025-07-12,completed,NaN,bank_link,parcel_locker,3.49,21.48,2025-07-15,2025-07-15,27.99,-10.0
2407,2408,1598,48866,2024-12-21 15:07:54,2024-12-21,completed,NaN,pay_later,courier,0.00,104.98,2024-12-27,2024-12-27,114.98,-10.0
2340,2341,2072,47618,2024-12-16 08:45:12,2024-12-16,completed,NaN,card,parcel_locker,0.00,99.97,2024-12-17,2024-12-17,104.97,-5.0
751,752,741,<NA>,2024-06-24 17:12:00,2024-06-24,completed,NaN,bank_link,courier,0.00,64.99,2024-06-26,2024-06-26,69.99,-5.0
8982,8983,6134,<NA>,2026-06-29 13:10:17,2026-06-29,completed,NaN,card,parcel_locker,0.00,144.98,2026-07-01,2026-07-01,149.98,-5.0
6839,6840,4400,140045,2026-01-06 19:42:07,2026-01-06,completed,NaN,card,parcel_locker,0.00,81.88,2026-01-09,2026-01-09,86.88,-5.0


**Answer**: I could not find a reason in the data. In all these orders 'order_total' is lower than the 'sum of lines + shipping', by a round amount, and there is no coupon. It looks like manual compensations from customer service (damaged product, wrong item and etc...). For revenue I will use order_total, because this is what the customer actually paid.

### Other differences (not 0.01 € and not round)

In [24]:
other_differences = not_matching_tolerance[~(round(round(not_matching_tolerance["difference"], 0), 2) == round(not_matching_tolerance["difference"], 2))]
print(f"Other differences: {len(other_differences)}")
other_differences["diff_total_share"] = other_differences["difference"] / (other_differences["total_paid"] + other_differences["shipping_fee"])
display(other_differences.sort_values("diff_total_share"))

Other differences: 11


,order_id,customer_id,session_id,order_ts,order_date_local,status,coupon_code,payment_method,delivery_method,shipping_fee,order_total,promised_date,delivered_date,total_paid,difference,diff_total_share
5557,5558,4080,<NA>,2025-10-13 19:21:07,2025-10-13,completed,NaN,card,courier,5.49,24.48,2025-10-15,2025-10-15,189.90,-170.91,-0.874712
5515,5516,1271,114965,2025-10-10 20:12:01,2025-10-10,completed,NaN,paypal,courier,0.00,98.97,2025-10-14,2025-10-14,368.88,-269.91,-0.731701
5707,5708,2962,118941,2025-10-28 05:21:59,2025-10-28,completed,NaN,bank_link,parcel_locker,0.00,64.97,2025-10-29,2025-10-29,199.88,-134.91,-0.674955
7369,7370,853,150067,2026-02-20 15:58:39,2026-02-20,completed,NaN,bank_link,courier,5.49,31.02,2026-02-24,2026-02-24,28.98,-3.45,-0.100087
9139,9140,5171,186845,2026-07-11 18:38:18,2026-07-11,completed,NaN,bank_link,parcel_locker,3.49,44.51,2026-07-14,2026-07-14,45.97,-4.95,-0.100081
7227,7228,5417,147023,2026-02-06 09:40:24,2026-02-06,completed,AFF10,bank_link,parcel_locker,3.49,36.33,2026-02-07,2026-02-07,36.88,-4.04,-0.100074
9793,9794,7196,200161,2026-08-29 09:29:02,2026-08-29,completed,NaN,card,courier,5.49,27.88,2026-09-01,2026-09-01,25.49,-3.10,-0.100065
1774,1775,1628,37163,2024-11-03 14:53:39,2024-11-03,completed,NaN,card,parcel_locker,2.99,29.68,2024-11-05,2024-11-05,29.99,-3.30,-0.100061
8869,8870,6127,180674,2026-06-20 07:59:21,2026-06-20,completed,NaN,bank_link,parcel_locker,4.49,38.23,2026-06-26,2026-06-27,37.99,-4.25,-0.100047
7957,7958,6002,162085,2026-04-09 16:01:47,2026-04-09,completed,NaN,card,courier,0.00,116.06,2026-04-13,2026-04-13,128.96,-12.90,-0.100031


**Answer:** 11 orders are left. They make 2 groups:

- **8 orders:** `order_total` is lower by exactly 10 % of (lines + shipping). There is no coupon that explains it. This looks like a manual 10 % compensation.
- **3 orders (5516, 5558, 5708):** the difference is very large. One line has quantity 10, but `order_total` fits quantity 1. The quantity is wrong (DQ-12). This also answers question B2.

**Final answer:** No. 44 orders don't match. I split them into 4 groups:

| Group | Orders | What I found | Decision |
|---|---|---|---|
| 0.01 € off | 17 | Rounding of lines that end in half a cent | Allow 0.01 € difference |
| Round amount (2, 5, 10 €...) | 16 | `order_total` is lower, no coupon, lines look fine | Looks like manual compensation |
| About 10 % of the order | 8 | `order_total` is lower by 10 % of (lines + shipping) | Looks like manual compensation |
| Very large difference | 3 | One line has quantity 10, but `order_total` fits quantity 1 | Quantity is wrong (DQ-12), fix to 1 |

- The 24 compensation orders (16 + 8) are not a data error. `order_total` is what the customer really paid, so I use it for revenue.
- The 3 large ones also answer question B2: the large quantities do not match `order_total`.
- I can't see the reason for the compensations in the data. In a real job I would ask customer service.

### B3. Why do purchase events not match orders (DQ-15)?